In [ ]:
import mujoco
import torch
import mediapy

from mppi_controller import MPPIController
from dynamical_systems.cart_pole import CartPole

import helper

## Cart pole

In [ ]:
R = 1
lambda_S = 10
K = 50                      # K rollouts
horizon = 1.0               # 1s
dt = 0.05                   # horizon / deltat = N

disturbance_param = 500     # 1/sqrt(rho)
N = int(horizon / dt)       # [t0, 1s] horizon

state_dim = 4               # x, x_dot, theta, theta_dot
control_input_dim = 1

In [ ]:
model = mujoco.MjModel.from_xml_path("mujoco_models/cartpole.xml")
data = mujoco.MjData(model)

mppi_controller = MPPIController(K, N, disturbance_param, dt, CartPole())



mujoco.mj_resetData(model, data)
mujoco.mj_forward(model, data)

frames = []
thetas = []


state = torch.tensor([data.qpos[0], data.qvel[0], data.qpos[1], data.qvel[1]])

with mujoco.Renderer(model, width=320, height=240) as renderer:
  
    while abs(abs(helper.normalize_angle(state[2].item())) - torch.pi) > 0.1:
      
        thetas.append(state[2])
        
        u = mppi_controller.get_action(state, control_input_dim)
        
        data.ctrl[0] = u[0].item()
        
        mujoco.mj_step(model, data)

        mujoco.mj_forward(model, data)


        renderer.update_scene(data)

        frames.append(renderer.render())

    media.show_video(frames, fps=10)